# Multi-factor screening

Apply Benjamini-Hochberg-Yekutieli (BHY) FDR control to a batch of
candidate factors. Demonstrates family partitioning — the behaviour
that is impossible to learn from docstrings alone.


## Factor type

This recipe uses `multi_factor.bhy(...)` over a list of
`FactorProfile` objects. Each profile is produced by
`evaluate(panel, cfg)` for any registered cell — screening is
factor-type-agnostic, but BHY families partition on the dispatch
key plus `forward_periods`.

Family key = `(scope, signal, metric, mode, forward_periods)`. Two
profiles share a family iff they share all five axes; cross-family
pooling silently inflates FDR and is gated by a runtime warning.

Literature: [Benjamini & Yekutieli (2001)](../reference/bibliography.md).


## Use this when

- You have ≥2 candidate factors and want type-I error control under
  multiple testing.
- Candidates are evaluable under the same `AnalysisConfig` (same
  scope, signal, metric, forward horizon). Mixed-cell batches still
  run but issue `RuntimeWarning` when most families collapse to
  size 1.
- You prefer FDR control (BHY) over family-wise error (Bonferroni)
  — appropriate when discoveries can tolerate a known false-positive
  rate and power matters.

## What it tests

Within each family, BHY step-up keeps profiles whose ranked
p-values satisfy `p_(k) ≤ q · k / (N · c(N))` where `c(N) =
Σ 1/i` (Benjamini-Yekutieli adjustment for arbitrary dependence).
Cross-family aggregation is *not* performed — that is the user's
responsibility and is deliberately out of scope.

## Output to read

1. `len(survivors)` vs `len(profiles)` — coarse hit rate.
2. Per-survivor `profile.primary_p` and `profile.config` — which
   factor cleared the family-adjusted bar.
3. Any emitted `RuntimeWarning` about singleton families — a sign
   the batch is mis-grouped (one profile per cell provides no FDR
   correction beyond a raw threshold).


## 1. Setup


In [1]:
from __future__ import annotations

import warnings

import factrix as fl
import numpy as np
import polars as pl
from factrix.preprocess.returns import compute_forward_return

print('factrix version:', fl.__version__)


factrix version: 0.7.0


## 2. Build a single-family batch

Five candidate factors, all under the same
`individual_continuous(IC, forward_periods=5)` cell — a *valid*
BHY input where the step-up actually controls FDR.

We start from one ground-truth factor and add increasing IID noise
to produce variants with varying signal strengths.


In [2]:
raw = fl.datasets.make_cs_panel(
    n_assets=100, n_dates=500, ic_target=0.08, seed=2024,
)
panel = compute_forward_return(raw, forward_periods=5)
cfg = fl.AnalysisConfig.individual_continuous(
    metric=fl.Metric.IC, forward_periods=5,
)

def with_noise(base: pl.DataFrame, scale: float, seed: int) -> pl.DataFrame:
    rng = np.random.default_rng(seed)
    return base.with_columns(
        pl.Series(
            'factor',
            base['factor'].to_numpy()
            + scale * rng.standard_normal(base.height),
        ),
    )

candidates = {
    f'variant_{i}': with_noise(panel, scale=0.5 + 0.3 * i, seed=100 + i)
    for i in range(5)
}
profiles = [fl.evaluate(p, cfg) for p in candidates.values()]
for name, prof in zip(candidates, profiles, strict=True):
    print(f'  {name:12s} primary_p={prof.primary_p:.4g}  verdict={prof.verdict()}')


  variant_0    primary_p=2.136e-39  verdict=pass
  variant_1    primary_p=4.052e-26  verdict=pass
  variant_2    primary_p=6.682e-22  verdict=pass
  variant_3    primary_p=3.987e-17  verdict=pass
  variant_4    primary_p=7.407e-14  verdict=pass


## 3. Apply BHY

`multi_factor.bhy` groups profiles by family key, runs the step-up
within each family, and returns survivors in input order across
families.


In [3]:
survivors = fl.multi_factor.bhy(profiles, threshold=0.05)
print(f'BHY survivors: {len(survivors)} / {len(profiles)}')
for prof in survivors:
    print(f'  primary_p={prof.primary_p:.4g}')


BHY survivors: 5 / 5
  primary_p=2.136e-39
  primary_p=4.052e-26
  primary_p=6.682e-22
  primary_p=3.987e-17
  primary_p=7.407e-14


## 4. Cross-family pitfall

Mixing cells in one `bhy()` call partitions into multiple families,
each size 1 in this construction. BHY on a singleton is identical to a raw
threshold and provides *no* FDR correction. factrix flags this
with `RuntimeWarning` so silent mis-grouping fails loud:


In [4]:
cfg_fm = fl.AnalysisConfig.individual_continuous(
    metric=fl.Metric.FM, forward_periods=5,
)
ev_raw = fl.datasets.make_event_panel(
    n_assets=80, n_dates=400, event_rate=0.02,
    post_event_drift_bps=15.0, signal_horizon=5, seed=42,
)
ev_panel = compute_forward_return(ev_raw, forward_periods=5)
cfg_caar = fl.AnalysisConfig.individual_sparse(forward_periods=5)

mixed = [
    fl.evaluate(panel, cfg),       # IC family
    fl.evaluate(panel, cfg_fm),    # FM family — different metric
    fl.evaluate(ev_panel, cfg_caar),  # sparse family — different signal
]
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    survivors_mixed = fl.multi_factor.bhy(mixed, threshold=0.05)
for w in caught:
    print(f'{w.category.__name__}: {w.message}')
print(f'\nsurvivors across mixed families: {len(survivors_mixed)}')



survivors across mixed families: 2


## 5. Sample-guard edge cases

Family partitioning makes BHY fragile to small-batch regimes. For
the broader `n_assets` × factory behaviour matrix see
[Guides § PANEL vs TIMESERIES](../guides/panel-timeseries.md);
for the BHY contract specifically see
[Guides § Batch screening](../guides/batch-screening.md).


In [5]:
print('multi_factor_screening: ok')


multi_factor_screening: ok
